# Overborrowing and Systemic Externalities in the Business Cycle Under Imperfect Information

## Step 0: Parameters

This notebook creates the parameter structure for the model. It follows the calibration in the paper:

- **Authors:** Juan Herreño (jherrenolopera@ucsd.edu) & Carlos Rondón Moreno (crondon@bcentral.cl)
- **Date:** 2025

### Contents
1. Import packages
2. Define the `Params` namedtuple
3. Create parameter factory function
4. Display Table 2 from the paper

In [1]:
# Import packages
import warnings
warnings.filterwarnings("ignore")

import jax
import jax.numpy as jnp
import numpy as np
from collections import namedtuple

jax.config.update("jax_enable_x64", True)

In [2]:
Params = namedtuple(
    "Params",
    (
        # Simulation control
        "nodes",        # Nodes in state-space for exogenous variables
        "grid_points",  # Grid points for debt
        "burn",         # Burn-in period
        "Tsim",         # Total simulation horizon (including burn-in)
        # IRF control
        "horizn",       # IRF horizon
        "init_debt",    # Initial debt index for IRFs (steady state)
        # Permanent component
        "rho_g",        # Autocorrelation of permanent growth
        "sigmag",       # Std. dev. of permanent growth rate
        "g",            # Mean growth rate (Garcia-Cicco et al. 2010)
        # VAR(1) for endowments
        "A",            # Autocorrelation matrix (3x3)
        "Sigma",        # Variance-covariance matrix (3x3)
        # Deep parameters
        "beta",         # Discount factor
        "kappa",        # Borrowing constraint constant
        "r",            # Annual interest rate
        "rho",          # Risk aversion coefficient
        "omega",        # Weight of tradables in CES
        "eta",          # Intratemporal elasticity of substitution
        # Grid dimensions
        "ytn",          # Grid points: transitory tradable
        "ynn",          # Grid points: transitory non-tradable
        "gn",           # Grid points: permanent component
        "gTn",          # Grid points: growth rate tradable output
        "gNn",          # Grid points: growth rate non-tradable output
        # Debt grid
        "bmin",         # Minimum debt
        "bmax",         # Maximum debt
        "bn",           # Number of debt grid points
        "b",            # Debt grid (jnp array)
        # Other
        "nstd",         # Std. devs. for crisis definition
        "window",       # Years around crises for event study
    )
)


def create_params(
    # Simulation control
    nodes: int = 10,
    grid_points: int = 101,
    burn: int = 1_000_000,
    # IRF control
    horizn: int = 21,
    # Permanent component parameters
    rho_g: float = 0.496779121757470,
    sigmag: float = 0.00327510105056510,
    g: float = None,  # defaults to log(1.0131)
    # Deep parameters
    beta: float = 0.83,
    kappa: float = 0.335,
    r: float = 0.04,
    rho: float = 2.0,
    omega: float = 0.31,
    eta: float = 0.83,
    # Debt grid bounds
    bmin: float = -1.4,
    bmax: float = -0.2,
    # Crisis parameters
    nstd: float = 1.0,
    window: int = 5,
) -> Params:
    """
    Create the full parameter structure for the overborrowing model.

    Returns a Params namedtuple with all calibrated values, grids,
    and VAR(1) matrices.
    """

    # Derived parameters
    if g is None:
        g = float(jnp.log(1.0131))  # Garcia-Cicco et al. (2010)

    Tsim = burn + 1_000_000  # Total simulation length
    init_debt = grid_points // 2  # IRFs start at steady state
    bn = grid_points

    # Grid dimensions (all equal to nodes)
    ytn = nodes
    ynn = nodes
    gn = nodes
    gTn = nodes
    gNn = nodes

    # VAR(1) autocorrelation matrix
    A = jnp.array([
        [0.734679129925539, -0.255320869154147, 0.0],
        [0.033652507003993,  0.417047371095781, 0.0],
        [0.0,                0.0,               rho_g]
    ])

    # VAR(1) variance-covariance matrix
    Sigma = jnp.array([
        [0.00461817366335938, 0.000379607811872954, 0.0],
        [0.000379607811872954, 0.00137117483970497, 0.0],
        [0.0, 0.0, sigmag]
    ])

    # Debt grid
    b = jnp.linspace(bmin, bmax, bn)

    return Params(
        nodes=nodes, grid_points=grid_points, burn=burn, Tsim=Tsim,
        horizn=horizn, init_debt=init_debt,
        rho_g=rho_g, sigmag=sigmag, g=g,
        A=A, Sigma=Sigma,
        beta=beta, kappa=kappa, r=r, rho=rho, omega=omega, eta=eta,
        ytn=ytn, ynn=ynn, gn=gn, gTn=gTn, gNn=gNn,
        bmin=bmin, bmax=bmax, bn=bn, b=b,
        nstd=nstd, window=window,
    )

In [3]:
def print_params(params):
    """
    Print all parameters of a Params namedtuple in a readable format.
    """
    print("Model Parameters:")
    print("=" * 60)
    for i, field_name in enumerate(Params._fields):
        value = getattr(params, field_name)
        print(f"{i:2d}  {field_name:15s}   ->  {value}")
    print("=" * 60)

In [6]:
# Create the baseline calibration
params = create_params()
print_params(params)

Model Parameters:
 0  nodes             ->  10
 1  grid_points       ->  101
 2  burn              ->  1000000
 3  Tsim              ->  2000000
 4  horizn            ->  21
 5  init_debt         ->  50
 6  rho_g             ->  0.49677912175747
 7  sigmag            ->  0.0032751010505651
 8  g                 ->  0.013014937077494544
 9  A                 ->  [[ 0.73467913 -0.25532087  0.        ]
 [ 0.03365251  0.41704737  0.        ]
 [ 0.          0.          0.49677912]]
10  Sigma             ->  [[0.00461817 0.00037961 0.        ]
 [0.00037961 0.00137117 0.        ]
 [0.         0.         0.0032751 ]]
11  beta              ->  0.83
12  kappa             ->  0.335
13  r                 ->  0.04
14  rho               ->  2.0
15  omega             ->  0.31
16  eta               ->  0.83
17  ytn               ->  10
18  ynn               ->  10
19  gn                ->  10
20  gTn               ->  10
21  gNn               ->  10
22  bmin              ->  -1.4
23  bmax             

## Table 2: Calibrated Parameters

Following Bianchi (2011), with $\beta$ and $\kappa$ calibrated to match Argentine data.

In [5]:
def display_table2(params):
    """
    Reproduce Table 2 from the paper.
    """
    rows = [
        ("r",            params.r,     "Interest Rate"),
        ("σ (rho)",      params.rho,   "Inverse of IES"),
        ("ε (eta)",      params.eta,   "Elasticity tradable / non-tradable"),
        ("ω (omega)",    params.omega, "Weight of tradable goods"),
        ("β (beta)",     params.beta,  "Discount factor"),
        ("κ (kappa)",    params.kappa, "Borrowing constraint constant"),
        ("μ_g (g)",      params.g,     "Avg. growth rate"),
    ]

    print("+" + "-" * 14 + "+" + "-" * 12 + "+" + "-" * 42 + "+")
    print(f"| {'Parameter':<12s} | {'Value':>10s} | {'Meaning':<40s} |")
    print("+" + "-" * 14 + "+" + "-" * 12 + "+" + "-" * 42 + "+")
    for name, val, desc in rows:
        print(f"| {name:<12s} | {val:10.4f} | {desc:<40s} |")
    print("+" + "-" * 14 + "+" + "-" * 12 + "+" + "-" * 42 + "+")


display_table2(params)

+--------------+------------+------------------------------------------+
| Parameter    |      Value | Meaning                                  |
+--------------+------------+------------------------------------------+
| r            |     0.0400 | Interest Rate                            |
| σ (rho)      |     2.0000 | Inverse of IES                           |
| ε (eta)      |     0.8300 | Elasticity tradable / non-tradable       |
| ω (omega)    |     0.3100 | Weight of tradable goods                 |
| β (beta)     |     0.8300 | Discount factor                          |
| κ (kappa)    |     0.3350 | Borrowing constraint constant            |
| μ_g (g)      |     0.0130 | Avg. growth rate                         |
+--------------+------------+------------------------------------------+
